# Nifty 50 Simulator — Data Layer Tests
Run each cell to validate the engine before touching the dashboard.

In [2]:
# Setup — run this first
import sys
sys.path.insert(0, '..')  # add project root to path
import pandas as pd
pd.set_option('display.float_format', '{:,.2f}'.format)
print('Setup OK')

Setup OK


## 1. Constituents — Static Data

In [18]:
from data.constituents import CONSTITUENTS, get_constituent_df

print(f'Total constituents loaded: {len(CONSTITUENTS)}')
print(f'Total weight sum: {sum(v["weight_pct"] for v in CONSTITUENTS.values()):.2f}%')

df = get_constituent_df()
print('\nTop 10 by weight:')
print(df[['name','sector','weight_pct','div_yield']].sort_values('weight_pct', ascending=False).head(10))

ImportError: cannot import name 'CONSTITUENTS' from 'data.constituents' (/Users/mohamedabdellahi/Desktop/0.Trust the process/0.projects/deltaone/index-rebal/data/constituents.py)

## 2. Live Prices (requires internet)

In [3]:
from data.constituents import get_live_prices, get_constituent_df

# Test with top 5 only to be fast
test_symbols = ['RELIANCE', 'HDFCBANK', 'TCS', 'ICICIBANK', 'INFY']
prices = get_live_prices(test_symbols)
print('Live prices:')
for sym, p in prices.items():
    print(f'  {sym}: ₹{p:,.2f}' if p else f'  {sym}: N/A')

TCS.NS: No data found for this date range, symbol may be delistedINFY.NS: No data found for this date range, symbol may be delisted
RELIANCE.NS: No data found for this date range, symbol may be delisted

HDFCBANK.NS: No data found for this date range, symbol may be delistedICICIBANK.NS: No data found for this date range, symbol may be delisted

Live prices:
  RELIANCE: N/A
  HDFCBANK: N/A
  TCS: N/A
  ICICIBANK: N/A
  INFY: N/A


In [19]:
import yfinance as yf

tk = yf.Ticker("RELIANCE.NS")
hist = tk.history(period="5d")
print(hist)
print(type(hist))
print(hist.shape)

                              Open     High      Low    Close    Volume  \
Date                                                                      
2026-03-09 00:00:00+05:30 1,375.00 1,429.40 1,370.00 1,424.00  24870822   
2026-03-10 00:00:00+05:30 1,430.60 1,431.50 1,400.60 1,408.80  18071259   
2026-03-11 00:00:00+05:30 1,424.90 1,434.00 1,383.60 1,390.20  21848087   
2026-03-12 00:00:00+05:30 1,390.00 1,410.90 1,381.10 1,392.20  20939959   
2026-03-13 00:00:00+05:30 1,385.20 1,400.80 1,378.40 1,380.70  17263343   

                           Dividends  Stock Splits  
Date                                                
2026-03-09 00:00:00+05:30       0.00          0.00  
2026-03-10 00:00:00+05:30       0.00          0.00  
2026-03-11 00:00:00+05:30       0.00          0.00  
2026-03-12 00:00:00+05:30       0.00          0.00  
2026-03-13 00:00:00+05:30       0.00          0.00  
<class 'pandas.core.frame.DataFrame'>
(5, 7)


In [3]:
from nsepython import nsefetch
import json

# Test — fetch Nifty 50 quote
url = "https://www.nseindia.com/api/equity-stockIndices?index=NIFTY%2050"
data = nsefetch(url)
print(type(data))
print(data.keys() if isinstance(data, dict) else data[:2])

<class 'dict'>
dict_keys(['name', 'advance', 'timestamp', 'data', 'metadata', 'marketStatus'])


In [4]:
# Cell — Test all constituent prices (single API call!)
from data.futures import get_live_constituent_prices
prices = get_live_constituent_prices()
print(f"Got {len(prices)} prices")
for sym, p in list(prices.items())[:5]:
    print(f"  {sym}: ₹{p:,.2f}")

Got 51 prices
  NIFTY 50: ₹23,151.10
  LT: ₹3,445.00
  HDFCBANK: ₹816.70
  RELIANCE: ₹1,382.10
  ICICIBANK: ₹1,255.00


In [5]:
# Cell 1 — Test dynamic constituents
from data.constituents import get_constituents
cs = get_constituents()
print(f"Total constituents: {len(cs)}")
for c in cs[:5]:
    print(f"  {c['symbol']:15} ₹{c['last_price']:>8,.2f}   {c['weight_pct']:.2f}%")

Total constituents: 50
  HDFCBANK        ₹  816.70   11.83%
  RELIANCE        ₹1,382.10   8.83%
  ICICIBANK       ₹1,255.00   8.49%
  BHARTIARTL      ₹1,803.00   4.76%
  INFY            ₹1,250.40   4.15%


In [6]:
# Cell 2 — Test as DataFrame
from data.constituents import get_constituent_df
df = get_constituent_df()
print(df[['name', 'sector', 'weight_pct', 'last_price', 'div_yield']].head(10))

                                   name  \
symbol                                    
HDFCBANK              HDFC Bank Limited   
RELIANCE    Reliance Industries Limited   
ICICIBANK            ICICI Bank Limited   
BHARTIARTL        Bharti Airtel Limited   
INFY                    Infosys Limited   
SBIN                State Bank of India   
LT              Larsen & Toubro Limited   
AXISBANK              Axis Bank Limited   
ITC                         ITC Limited   
KOTAKBANK   Kotak Mahindra Bank Limited   

                                              sector  weight_pct  last_price  \
symbol                                                                         
HDFCBANK                         Private Sector Bank       11.83      816.70   
RELIANCE                      Refineries & Marketing        8.83    1,382.10   
ICICIBANK                        Private Sector Bank        8.49    1,255.00   
BHARTIARTL  Telecom - Cellular & Fixed line services        4.76    1,803.00   
INF

In [7]:
#  — Test prices dict
from data.constituents import get_prices_dict
prices = get_prices_dict()
print(f"Got {len(prices)} prices")
print(prices)

Got 50 prices
{'HDFCBANK': 816.7, 'RELIANCE': 1382.1, 'ICICIBANK': 1255.0, 'BHARTIARTL': 1803.0, 'INFY': 1250.4, 'SBIN': 1046.0, 'LT': 3445.0, 'AXISBANK': 1199.1, 'ITC': 301.9, 'KOTAKBANK': 367.1, 'M&M': 2940.0, 'TCS': 2412.0, 'BAJFINANCE': 856.5, 'HINDUNILVR': 2161.8, 'SUNPHARMA': 1802.0, 'NTPC': 384.05, 'TITAN': 4072.5, 'MARUTI': 12615.0, 'BEL': 439.0, 'ETERNAL': 217.0, 'TATASTEEL': 183.01, 'SHRIRAMFIN': 1005.9, 'HCLTECH': 1329.1, 'POWERGRID': 301.0, 'HINDALCO': 910.9, 'ULTRACEMCO': 10744.0, 'COALINDIA': 466.5, 'JSWSTEEL': 1120.0, 'ONGC': 264.05, 'ADANIPORTS': 1368.8, 'BAJAJFINSV': 1750.8, 'ASIANPAINT': 2194.1, 'BAJAJ-AUTO': 8875.0, 'GRASIM': 2570.0, 'INDIGO': 4162.0, 'EICHERMOT': 6765.0, 'NESTLEIND': 1202.1, 'SBILIFE': 1908.8, 'TECHM': 1332.0, 'DRREDDY': 1290.0, 'APOLLOHOSP': 7545.0, 'TRENT': 3500.0, 'JIOFIN': 235.85, 'CIPLA': 1315.1, 'MAXHEALTH': 991.0, 'TATACONSUM': 1082.0, 'HDFCLIFE': 624.6, 'TMPV': 314.2, 'WIPRO': 197.75, 'ADANIENT': 1960.0}


## 3. Futures — Expiry Dates & Spot Price

In [9]:
from data.futures import get_expiry_dates, get_days_to_expiry, get_nifty_spot

expiries = get_expiry_dates(3)
print('Next 3 Nifty expiry dates:')
for e in expiries:
    print(f'  {e.strftime("%B %d, %Y")} — {get_days_to_expiry(e)} days')

spot = get_nifty_spot()
print(f'\nNifty spot: {spot:,.2f}' if spot else '\nNifty spot: N/A')

Next 3 Nifty expiry dates:
  March 26, 2026 — 11 days
  April 30, 2026 — 46 days
  May 28, 2026 — 74 days

Nifty spot: 23,151.10


## 4. Fair Value Calculation

In [12]:
from data.constituents import get_constituent_df, get_prices_dict
from data.futures import get_nifty_spot, get_days_to_expiry
from data.index_math import compute_dividend_drag, compute_fair_value

df     = get_constituent_df()
prices = get_prices_dict()
spot   = get_nifty_spot()
days   = get_days_to_expiry()

print(f"Spot: {spot:,.2f}, Days to expiry: {days}")
# ← You provide these two, no assumptions
# either we use live prices or mock prices, but we need both spot and days to expiry to compute fair value
FUTURES_PRICE  = spot*0.998   # type the real futures price ## 
RISK_FREE_RATE = 0.065     # RBI repo rate

drag = compute_dividend_drag(df, prices, spot, days)
fv   = compute_fair_value(spot, days, drag, FUTURES_PRICE, RISK_FREE_RATE)

print(f"Spot          : {fv.spot:,.2f}")
print(f"Cost of carry : +{fv.cost_of_carry:.2f} pts")
print(f"Dividend drag : -{fv.dividend_drag:.2f} pts")
print(f"Fair value    : {fv.fair_value:,.2f}")
print(f"Basis         : {fv.basis:+.2f} pts")
print(f"Basis ann.    : {fv.basis_annualized:+.4f}%")

Spot: 23,151.10, Days to expiry: 11
Spot          : 23,151.10
Cost of carry : +45.35 pts
Dividend drag : -7.51 pts
Fair value    : 23,188.94
Basis         : -84.15 pts
Basis ann.    : -12.0605%


In [13]:
# Breakdown of dividend drag by constituent
time_fraction = days / 365.0
rows = []

for symbol, row in df.iterrows():
    if symbol not in prices:
        continue
    drag_i = spot * (row["weight_pct"] / 100) * (row["div_yield"] / 100) * time_fraction
    rows.append({
        "symbol":      symbol,
        "name":        row["name"],
        "weight_%":    round(row["weight_pct"], 2),
        "div_yield_%": round(row["div_yield"], 2),
        "drag_pts":    round(drag_i, 4),
        "drag_%total": round(drag_i / drag * 100, 2),
    })

import pandas as pd
breakdown = pd.DataFrame(rows).sort_values("drag_pts", ascending=False)
print(breakdown.to_string(index=False))
print(f"\nTOTAL: {breakdown['drag_pts'].sum():.4f} pts")

    symbol                                          name  weight_%  div_yield_%  drag_pts  drag_%total
  HDFCBANK                             HDFC Bank Limited     11.83         1.10      0.91        12.10
      INFY                               Infosys Limited      4.15         2.20      0.64         8.48
       ITC                                   ITC Limited      2.75         3.20      0.61         8.19
      SBIN                           State Bank of India      4.11         1.80      0.52         6.87
 ICICIBANK                            ICICI Bank Limited      8.49         0.80      0.47         6.32
 COALINDIA                            Coal India Limited      1.00         6.50      0.45         6.06
 POWERGRID       Power Grid Corporation of India Limited      1.29         3.80      0.34         4.56
   HCLTECH                      HCL Technologies Limited      1.33         3.50      0.32         4.32
      NTPC                                  NTPC Limited      1.72       

## 5. Module 1 — Dividend Shock

In [14]:
from data.index_math import simulate_dividend_shock

# ── YOUR INPUTS ───────────────────────────────────────────────
SYMBOL            = "HDFCBANK"   # stock you want to shock
SHOCKED_DIV_YIELD = 3.0          # your scenario yield in %
FUTURES_PRICE     = 23100.0      # real futures price
RISK_FREE_RATE    = 0.065        # RBI repo rate
# ─────────────────────────────────────────────────────────────

result = simulate_dividend_shock(
    symbol            = SYMBOL,
    shocked_div_yield = SHOCKED_DIV_YIELD,
    df                = df,
    prices            = prices,
    index_level       = spot,
    days_to_expiry    = days,
    market_futures    = FUTURES_PRICE,
    risk_free_rate    = RISK_FREE_RATE,
)

print(f"Stock          : {result.stock_name} ({result.weight_pct:.2f}% of index)")
print(f"Div yield      : {result.base_div_yield:.2f}% → {result.shocked_div_yield:.2f}%")
print(f"Div drag Δ     : {result.delta_div_points:+.4f} pts")
print(f"Fair value     : {result.base_fair_value:.2f} → {result.shocked_fair_value:.2f}")
print(f"FV Δ           : {result.delta_fair_value:+.2f} pts")
print(f"P&L per lot    : ₹{result.delta_per_lot:+,.0f}")

Stock          : HDFC Bank Limited (11.83% of index)
Div yield      : 1.10% → 3.00%
Div drag Δ     : +1.5683 pts
Fair value     : 23188.94 → 23187.38
FV Δ           : -1.56 pts
P&L per lot    : ₹-117


## 6. Module 2 — Constituent Swap

In [16]:
from data.index_math import simulate_constituent_swap
from data.constituents import CANDIDATE_ADDITIONS

# Simulate: Remove DRREDDY, Add TRENT
trent = CANDIDATE_ADDITIONS['TRENT']
result = simulate_constituent_swap(
    symbol_out    = 'DRREDDY',
    symbol_in     = 'TRENT',
    price_in      = 6200.0,
    iwf_in        = trent['iwf'],
    shares_ff_in  = trent['shares_ff'],
    constituents_df = df,
    prices        = mock_prices,
    index_level   = INDEX_LEVEL,
    days_to_expiry = DAYS,
    market_futures = MARKET_FUTURES,
    tracker_aum_bn = 150.0,
)

print(f'OUT: {result.stock_out}  →  IN: {result.stock_in}')
print(f'Index level:     {result.old_index_level:.2f} → {result.new_index_level:.2f} ({result.delta_index_points:+.2f} pts)')
print(f'Futures FV:      {result.old_fair_value:.2f} → {result.new_fair_value:.2f} ({result.delta_future_points:+.2f} pts)')
print(f'Tracker must BUY  {result.buy_stock_in_shares:,.0f} shares of {result.stock_in}  = ₹{result.buy_stock_in_value/1e7:.1f} Cr')
print(f'Tracker must SELL {result.sell_stock_out_shares:,.0f} shares of {result.stock_out} = ₹{result.sell_stock_out_value/1e7:.1f} Cr')

KeyError: 'shares_ff'

## 7. Sensitivity Table (Rate vs Fair Value)

In [ ]:
from data.index_math import sensitivity_table

tbl = sensitivity_table(INDEX_LEVEL, DAYS, drag, MARKET_FUTURES)
print('Sensitivity of Fair Value to Risk-Free Rate:')
print(tbl.to_string(index=False))